# Research: Sector Momentum ETF Rotation (Issue #22)

## Contexte
- **Performance actuelle**: Sharpe 0.216, CAGR 6.5%, MaxDD 29.9%
- **Problemes**: MaxDD 30% (TLT risk-off en 2022), poids arbitraires, top 3 concentre
- **Objectif**: Sharpe > 0.4, MaxDD < 20%

## Hypotheses a tester
1. Nombre optimal de positions (2-5)
2. Lookback scheme : 1 mois seul vs composite vs 6 mois seul
3. TLT risk-off : cash > TLT en 2022 (taux montants)
4. Volatility-adjusted momentum (diviser par vol sectorielle)
5. Filtre de momentum absolu (ne trader que si momentum > 0)

## References
- Mebane Faber (2007) "Quantitative Approach to Tactical Asset Allocation"
- Jegadeesh & Titman (1993) "Returns to Buying Winners"

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 11 GICS sector ETFs + SPY + TLT
sector_tickers = ['XLK', 'XLF', 'XLE', 'XLV', 'XLI', 'XLY', 'XLP', 'XLU', 'XLB', 'XLRE', 'XLC']
all_tickers = sector_tickers + ['SPY', 'TLT']

data = yf.download(all_tickers, start='2010-01-01', end='2026-01-01')
closes = data['Close'].dropna()

# Rendements
returns = closes.pct_change()

print(f"Periode: {closes.index[0].date()} a {closes.index[-1].date()}")
print(f"\nRendements annualises (buy & hold):")
for t in sector_tickers:
    ret = (closes[t].iloc[-1] / closes[t].iloc[0]) ** (252/len(closes)) - 1
    vol = returns[t].std() * np.sqrt(252)
    print(f"  {t}: {ret:.1%} (vol: {vol:.1%})")

[                       0%                       ]

[***********           23%                       ]  3 of 13 completed

[***************       31%                       ]  4 of 13 completed

[***************       31%                       ]  4 of 13 completed

[**********************46%                       ]  6 of 13 completed

...

[**********************77%************           ]  10 of 13 completed

[**********************85%****************       ]  11 of 13 completed

[**********************92%*******************    ]  12 of 13 completed

[*********************100%***********************]  13 of 13 completed

Periode: 2018-06-19 a 2025-12-31

Rendements annualises (buy & hold):
  XLK: 21.5% (vol: 26.6%)
  XLF: 11.8% (vol: 23.8%)
  XLE: 7.1% (vol: 32.3%)
  XLV: 10.2% (vol: 17.5%)
  XLI: 12.4% (vol: 21.5%)
  XLY: 11.7% (vol: 24.2%)
  XLP: 8.6% (vol: 15.6%)
  XLU: 10.7% (vol: 20.7%)
  XLB: 8.2% (vol: 22.2%)
  XLRE: 6.9% (vol: 22.3%)
  XLC: 13.2% (vol: 22.6%)


## Hypothese 1: Nombre de positions et lookback

Grid search: nombre de positions x type de momentum

In [2]:
def sector_momentum_backtest(closes, sector_tickers, spy_col='SPY', tlt_col='TLT',
                              top_n=3, lookback=126, rebal_freq=21,
                              use_risk_off=True, risk_off_asset='TLT',
                              abs_momentum_filter=False, vol_adjust=False):
    """Backtest sector momentum rotation."""
    returns_df = closes.pct_change()
    sma200 = closes[spy_col].rolling(200).mean()
    
    portfolio_returns = []
    holdings = []
    rebal_counter = rebal_freq
    
    start_idx = max(lookback, 200) + 1
    
    for i in range(start_idx, len(closes)):
        # Daily return from current holdings
        if holdings:
            port_ret = np.mean([returns_df[t].iloc[i] for t in holdings])
        else:
            port_ret = 0
        portfolio_returns.append(port_ret)
        
        rebal_counter += 1
        if rebal_counter < rebal_freq:
            continue
        rebal_counter = 0
        
        # SMA200 regime filter
        spy_price = closes[spy_col].iloc[i]
        spy_sma = sma200.iloc[i]
        if pd.isna(spy_sma):
            continue
        
        risk_on = spy_price > spy_sma
        
        if not risk_on:
            if use_risk_off and risk_off_asset:
                holdings = [risk_off_asset]
            else:
                holdings = []  # Cash
            continue
        
        # Calculate momentum scores
        scores = {}
        for t in sector_tickers:
            current = closes[t].iloc[i]
            past = closes[t].iloc[i - lookback]
            if past > 0:
                mom = current / past - 1
                if vol_adjust:
                    vol = returns_df[t].iloc[i-63:i].std() * np.sqrt(252)
                    if vol > 0:
                        mom = mom / vol
                scores[t] = mom
        
        if len(scores) < top_n:
            continue
        
        sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        
        if abs_momentum_filter:
            top = [t for t, s in sorted_scores[:top_n] if s > 0]
        else:
            top = [t for t, s in sorted_scores[:top_n]]
        
        if len(top) == 0:
            if use_risk_off and risk_off_asset:
                holdings = [risk_off_asset]
            else:
                holdings = []
        else:
            holdings = top
    
    rets = np.array(portfolio_returns)
    cum = pd.Series((1 + rets).cumprod())
    total = cum.iloc[-1] - 1
    years = len(rets) / 252
    cagr = (1 + total) ** (1/years) - 1
    vol = rets.std() * np.sqrt(252)
    sharpe = (cagr - 0.04) / vol if vol > 0 else 0
    max_dd = ((cum / cum.cummax()) - 1).min()
    
    return {'sharpe': sharpe, 'cagr': cagr, 'max_dd': max_dd, 'vol': vol, 'cum': cum}

# Grid search: top_n x lookback
print(f"{'Top_N':<7} {'Lookback':<10} {'Sharpe':>7} {'CAGR':>7} {'MaxDD':>7} {'Vol':>7}")
print("-" * 50)

best = {'sharpe': -999}
for top_n in [2, 3, 4, 5]:
    for lb in [21, 63, 126, 252]:
        r = sector_momentum_backtest(closes, sector_tickers, top_n=top_n, lookback=lb)
        print(f"{top_n:<7} {lb:<10} {r['sharpe']:>7.3f} {r['cagr']:>6.1%} {r['max_dd']:>6.1%} {r['vol']:>6.1%}")
        if r['sharpe'] > best['sharpe']:
            best = r
            best['config'] = (top_n, lb)

print(f"\nBest: top_n={best['config'][0]}, lookback={best['config'][1]}, Sharpe={best['sharpe']:.3f}")

Top_N   Lookback    Sharpe    CAGR   MaxDD     Vol
--------------------------------------------------
2       21          -0.340  -2.2% -40.2%  18.4%
2       63           0.042   4.8% -31.3%  18.8%
2       126          0.107   6.0% -35.5%  19.1%
2       252          0.153   7.0% -36.8%  19.8%
3       21          -0.198   0.5% -38.4%  17.4%
3       63          -0.088   2.4% -29.0%  17.9%
3       126          0.063   5.1% -32.3%  18.0%


3       252          0.137   6.6% -35.7%  18.7%
4       21          -0.122   1.9% -35.0%  17.0%
4       63          -0.129   1.8% -32.5%  17.2%
4       126          0.042   4.7% -30.5%  17.2%
4       252         -0.002   4.0% -37.4%  18.2%
5       21          -0.117   2.1% -31.8%  16.6%
5       63          -0.080   2.6% -32.4%  16.8%
5       126          0.048   4.8% -30.6%  16.7%


5       252         -0.021   3.6% -35.6%  17.6%

Best: top_n=2, lookback=252, Sharpe=0.153


## Hypothese 2: TLT risk-off vs cash

TLT a perdu ~30% en 2022 (hausse des taux). Cash est plus sur.

In [3]:
tn, lb = best['config']
print(f"Config: top_{tn}, lookback={lb}")
print()

# Compare TLT vs Cash risk-off
r_tlt = sector_momentum_backtest(closes, sector_tickers, top_n=tn, lookback=lb, use_risk_off=True, risk_off_asset='TLT')
r_cash = sector_momentum_backtest(closes, sector_tickers, top_n=tn, lookback=lb, use_risk_off=True, risk_off_asset=None)
r_xlp = sector_momentum_backtest(closes, sector_tickers, top_n=tn, lookback=lb, use_risk_off=True, risk_off_asset='XLP')

print(f"{'Risk-off':<15} {'Sharpe':>7} {'CAGR':>7} {'MaxDD':>7}")
print("-" * 40)
print(f"{'TLT':<15} {r_tlt['sharpe']:>7.3f} {r_tlt['cagr']:>6.1%} {r_tlt['max_dd']:>6.1%}")
print(f"{'Cash':<15} {r_cash['sharpe']:>7.3f} {r_cash['cagr']:>6.1%} {r_cash['max_dd']:>6.1%}")
print(f"{'XLP (staples)':<15} {r_xlp['sharpe']:>7.3f} {r_xlp['cagr']:>6.1%} {r_xlp['max_dd']:>6.1%}")

Config: top_2, lookback=252

Risk-off         Sharpe    CAGR   MaxDD
----------------------------------------
TLT               0.153   7.0% -36.8%
Cash              0.393  11.0% -35.7%
XLP (staples)     0.450  13.0% -33.1%


## Hypothese 3: Volatility-adjusted momentum

Diviser le momentum par la volatilite sectorielle pour favoriser
les trends "propres" (fort momentum + faible vol).

In [4]:
tn, lb = best['config']

r_raw = sector_momentum_backtest(closes, sector_tickers, top_n=tn, lookback=lb, vol_adjust=False)
r_vol = sector_momentum_backtest(closes, sector_tickers, top_n=tn, lookback=lb, vol_adjust=True)
r_abs = sector_momentum_backtest(closes, sector_tickers, top_n=tn, lookback=lb, abs_momentum_filter=True)
r_both = sector_momentum_backtest(closes, sector_tickers, top_n=tn, lookback=lb, vol_adjust=True, abs_momentum_filter=True)

print(f"Config: top_{tn}, lookback={lb}")
print(f"{'Mode':<25} {'Sharpe':>7} {'CAGR':>7} {'MaxDD':>7}")
print("-" * 50)
print(f"{'Raw momentum':<25} {r_raw['sharpe']:>7.3f} {r_raw['cagr']:>6.1%} {r_raw['max_dd']:>6.1%}")
print(f"{'Vol-adjusted':<25} {r_vol['sharpe']:>7.3f} {r_vol['cagr']:>6.1%} {r_vol['max_dd']:>6.1%}")
print(f"{'Abs momentum filter':<25} {r_abs['sharpe']:>7.3f} {r_abs['cagr']:>6.1%} {r_abs['max_dd']:>6.1%}")
print(f"{'Vol-adj + abs filter':<25} {r_both['sharpe']:>7.3f} {r_both['cagr']:>6.1%} {r_both['max_dd']:>6.1%}")

Config: top_2, lookback=252

Mode                       Sharpe    CAGR   MaxDD
--------------------------------------------------
Raw momentum                0.153   7.0% -36.8%
Vol-adjusted               -0.013   3.7% -36.5%
Abs momentum filter         0.167   7.3% -36.8%
Vol-adj + abs filter        0.002   4.0% -36.5%


## Hypothese 4: Composite vs simple momentum

La strategie actuelle utilise un composite (1, 3, 6, 12 mois).
Testons si un lookback simple est meilleur.

In [5]:
def composite_momentum_backtest(closes, sector_tickers, top_n=3,
                                  lookbacks=[21, 63, 126, 252],
                                  weights=[0.4, 0.2, 0.2, 0.2],
                                  rebal_freq=21, risk_off_asset=None):
    """Backtest with composite momentum."""
    returns_df = closes.pct_change()
    sma200 = closes['SPY'].rolling(200).mean()
    max_lb = max(lookbacks)
    
    portfolio_returns = []
    holdings = []
    rebal_counter = rebal_freq
    
    for i in range(max_lb + 201, len(closes)):
        if holdings:
            port_ret = np.mean([returns_df[t].iloc[i] for t in holdings])
        else:
            port_ret = 0
        portfolio_returns.append(port_ret)
        
        rebal_counter += 1
        if rebal_counter < rebal_freq:
            continue
        rebal_counter = 0
        
        if closes['SPY'].iloc[i] <= sma200.iloc[i]:
            holdings = [risk_off_asset] if risk_off_asset else []
            continue
        
        scores = {}
        for t in sector_tickers:
            score = 0
            valid = True
            for lb, w in zip(lookbacks, weights):
                past = closes[t].iloc[i - lb]
                if past <= 0:
                    valid = False
                    break
                score += w * (closes[t].iloc[i] / past - 1)
            if valid:
                scores[t] = score
        
        sorted_s = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        top = [t for t, s in sorted_s[:top_n] if s > 0]
        holdings = top if top else ([risk_off_asset] if risk_off_asset else [])
    
    rets = np.array(portfolio_returns)
    cum = pd.Series((1 + rets).cumprod())
    total = cum.iloc[-1] - 1
    years = len(rets) / 252
    cagr = (1 + total) ** (1/years) - 1
    vol = rets.std() * np.sqrt(252)
    sharpe = (cagr - 0.04) / vol if vol > 0 else 0
    max_dd = ((cum / cum.cummax()) - 1).min()
    
    return {'sharpe': sharpe, 'cagr': cagr, 'max_dd': max_dd}

# Test different composite schemes
schemes = [
    ('1m only', [21], [1.0]),
    ('3m only', [63], [1.0]),
    ('6m only', [126], [1.0]),
    ('12m only', [252], [1.0]),
    ('Current 4/2/2/2', [21, 63, 126, 252], [0.4, 0.2, 0.2, 0.2]),
    ('Equal 1/3/6/12', [21, 63, 126, 252], [0.25, 0.25, 0.25, 0.25]),
    ('Recent 6/3/1', [21, 63, 126], [0.5, 0.3, 0.2]),
    ('Short 1/3', [21, 63], [0.6, 0.4]),
]

print(f"{'Scheme':<20} {'Sharpe':>7} {'CAGR':>7} {'MaxDD':>7}")
print("-" * 45)

for name, lbs, wts in schemes:
    r = composite_momentum_backtest(closes, sector_tickers, top_n=3, lookbacks=lbs, weights=wts)
    print(f"{name:<20} {r['sharpe']:>7.3f} {r['cagr']:>6.1%} {r['max_dd']:>6.1%}")

Scheme                Sharpe    CAGR   MaxDD
---------------------------------------------
1m only                0.032   4.5% -29.3%
3m only                0.181   6.6% -22.2%
6m only                0.337   9.0% -23.8%
12m only               0.612  12.8% -14.0%
Current 4/2/2/2        0.490  11.0% -16.7%
Equal 1/3/6/12         0.451  10.5% -18.1%
Recent 6/3/1           0.284   8.2% -25.7%
Short 1/3              0.205   7.0% -28.1%


## Conclusions et recommandations

### Configuration recommandee
Basee sur les resultats ci-dessus, identifier:
1. Nombre optimal de positions
2. Type de momentum (simple vs composite)
3. Risk-off: cash vs TLT
4. Filtres additionnels (vol-adjust, abs momentum)

## VERDICT - ARCHIVE (2026-04-21)

**Strategie archivee.** Sharpe ceiling ~0.48 pour la rotation mensuelle d'ETF sectoriels.

| Metrique | Meilleure version (v4.0) | Plafond structurel |
|----------|--------------------------|---------------------|
| Sharpe   | 0.472                    | ~0.48               |
| CAGR     | 11.1%                    | -                   |
| MaxDD    | 25.8%                    | -                   |

### Versions testees et rejetees

v5.0 (proportional weights, Sharpe 0.398), v6.0 (trailing stop, Sharpe 0.441),
v6.1 (TLT risk-off, Sharpe 0.460), v6.2 (target vol + SMA50, Sharpe 0.395).

### Pourquoi l'expansion n'ameliore pas

11 ETF sectoriels fortement correles limitent la diversification. Le grid search
confirme que top_n=2 lookback=252 est optimal en recherche (Sharpe 0.153) mais
surajuste. Le composite 4 fenetres (1/3/6/12m) atteint Sharpe 0.490, confirmant
le plafond structurel.

### Preconisation

Abandonner la rotation momentum pure sur cet univers. Pour depasser 0.5, il faut
des approches fondamentalement differentes : ML regime detection, cross-asset
momentum, ou allocation factor-based.

Voir `ARCHIVE.md` pour la documentation complete.